In [1]:
import pandas as pd
import numpy as np
import random

In [2]:
Data = pd.read_parquet('train.parquet')

In [3]:
DataItems = pd.read_parquet("benchmark_items.parquet")

<p style="text-align: center;"> 1. Идейный подход</p>

<p style="text-indent: 50px;">Исходный датасет содержит в себе лишь те примеры, которые соответствуют выбору пользователя, тем самым нет объявлений, которые бы давали метку как неподходящую. Также нетрудно заметить, что нет смысла рассматривать примеры из другой категории (пользователь делает запрос по категории А - ожидаемо, что предлагаемые пользователю объявления из категории А, а не из какой-либо другой). 

<p style="text-indent: 50px;">Также стоит отметить, что если перебирать пары (запрос, объявление), то  предсказание будет достаточно долгим в виду того, что придётся перебрать все имеющиеся объявления, потому модели для предсказания будут подаваться лишь те примеры, которые соответствуют категории запроса. Помимо этого, я дополнил обучающую выборку примерами для каждого запроса теми примерами, которые дают метку 0 (объявление не подходит по запросу пользователя).

In [4]:
# Исходный датасет состоит исключительно из релевантных объявлений, метка класса 1 
Data["label"] = 1 

In [5]:
# Группируем объявления по подкатегориям: получаем словарь "подкатегория : список id-объявлений".
# Нужен был для отбора нерелевантных объявлений из той же подкатегории, что и релевантное.
# Используется только для обучения модели 
DictCategory = DataItems.groupby("item_microcat_id")["item_id"].agg(list).to_dict()

In [6]:
# Для воспроизводимости результатов зафиксируем random.seed
random.seed(42)

# Список id объявлений по каждой главной категории, без тех, что уже есть в train
train_items = set(Data["item_id"])

items_by_cat = {
    cat: list(set(group["item_id"]) - train_items)
    for cat, group in DataItems.groupby("item_category_id")
}

# Для каждого запроса берём 2 случайных объявления из его главной категории,
# которые пользователь не выбирал (label = 0)
def add_negatives(row, n=2):
    pool = items_by_cat.get(row["search_category"], [])
    if len(pool) < n:
        return []
    return random.sample(pool, n)

Data["negatives"] = Data.apply(add_negatives, axis=1)

# Разворачиваем списки в отдельные строки:
NewData = Data.explode("negatives").reset_index(drop=True)
NewData = NewData[NewData["negatives"].notna()].reset_index(drop=True)

item_cols = [c for c in NewData.columns if c.startswith("item_")]
NewData = NewData.drop(columns=item_cols)

NewData = NewData.merge(
    DataItems,
    left_on="negatives",
    right_on="item_id",
    how="left"
)
NewData = NewData.drop(columns=["negatives"])
NewData["label"] = 0

Data = Data.drop(columns=["negatives"])

# Объединяем примеры (label = 1) и (label = 0), затем перемешиваем.
TrainData = pd.concat([Data, NewData], ignore_index=True)
TrainData = TrainData.sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score
from xgboost import XGBClassifier 

### Текстовые признаки для обучения модели

**Задача.** Даны запрос $q$ и объявление с заголовком $t$ и описанием $d$. Нужно численно оценить их смысловую близость.

**Шаг 1. TF-IDF-векторизация.**

Каждый текст превращается в вектор размерности $V$ (число слов в словаре). Компонента вектора для слова $w$:

$$\text{tfidf}(w, d) = \text{tf}(w, d) \cdot \text{idf}(w)$$

где:

$$\text{tf}(w, d) = \frac{n_{w,d}}{\sum_{w'} n_{w',d}}$$

- нормированная частота слова $w$ в документе $d$ ($n_{w,d}$ - число вхождений);

$$\text{idf}(w) = \log \frac{N}{1 + |\{d : w \in d\}|}$$

- обратная частота ($N$ - общее число текстов).

**Почему IDF.** Слова, встречающиеся во всех текстах («и», «в», «услуга»), получают низкий вес. Редкие слова («автоподбор», «шале») - высокий. Это позволяет различать тексты.

**Шаг 2. Нормализация.**

Каждый вектор приводится к единичной длине:

$$\hat{v} = \frac{v}{\|v\|_2}$$

Это нужно, чтобы длина текста не влияла на близость.

**Шаг 3. Косинусное расстояние.**

Близость запроса $q$ и рассматриваемого текста из объявления $d$:

$$\cos(q, d) = \frac{q \cdot d}{\|q\|_2 \cdot \|d\|_2} = \hat{q} \cdot \hat{d}$$

После нормализации - это скалярное произведение. Значение от 0 (нет общих слов) до 1 (полное совпадение).

**Величины подсчёта:**

$\cos(q, \text{title})$ - совпадение запроса с заголовком.

$\cos(q, \text{description})$ - совпадение запроса с описанием.

**Параметры:** 

`min_df=5` отсекает слова, встречающиеся менее 5 раз (шум). `max_features=50000` ограничивает словарь по частоте.

**Примечание** 

TF-IDF проще и легче, чем эмбеддинги или предобученные языковые модели. Эмбеддинги дали бы лучшее качество на семантике (синонимы, перефраз), но требуют больше памяти и эффективнее работают на GPU. На моём ноутбуке это нецелесообразно — не хватит мощности и нет возможности задействовать GPU.



In [11]:
for col in ["search_query", "item_title_raw", "item_description_raw"]:
    TrainData[col] = TrainData[col].fillna("").astype(str)

vectorizer = TfidfVectorizer(
    lowercase=True,
    min_df=5,
    max_features=50000,
    dtype=np.float32,
)

all_texts = pd.concat([
    TrainData["search_query"],
    TrainData["item_title_raw"],
    TrainData["item_description_raw"]
]).unique()

vectorizer.fit(all_texts)

Q = normalize(vectorizer.transform(TrainData["search_query"]))
T = normalize(vectorizer.transform(TrainData["item_title_raw"]))
D = normalize(vectorizer.transform(TrainData["item_description_raw"]))

TrainData["tfidf_cosine_title"] = np.asarray(Q.multiply(T).sum(axis=1)).ravel()
TrainData["tfidf_cosine_desc"] = np.asarray(Q.multiply(D).sum(axis=1)).ravel()

del Q, T, D

In [20]:
# Сохраняю датасет на диск, чтобы не пересчитывать признаки при перезапуске.
TrainData.to_parquet("TrainData.parquet", index=False)

In [15]:
TrainData.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,...,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id,label,tfidf_cosine_title,tfidf_cosine_desc
0,перетяжка руля,659020,0,,114,"Маникюр и педикюр Широкая речка, Академ",19.0,4.894737,2000.000000000000000,86470,...,56.797485350000002,True,False,"Вид услуги Красота, здоровье Место оказания ус...",f8f8e3da38bd32dc,Доступную дату/время уточняйте в сообщениях\n\...,114,0,0.0,0.0
1,прокат костюмов,628970,0,"Вид услуги Праздники, мероприятия",114,Гадалка Таролог Маг Карты Таро Приворот привязка,12.0,4.666667,500.000000000000000,86460,...,53.888069150000000,False,False,"Вид услуги Обучение, курсы Место оказания услу...",c126e67114733c4b,РЕАЛЬНАЯ помощь ЯСНОВИДЯЩЕЙ в ДЕНЬ ОБРАЩЕНИЯ!\...,114,0,0.0,0.0
2,измельчитель веток,643700,0,,114,Земляные*Земельные работы*Канализация*Демонтаж,27.0,4.814815,333.000000000000000,1178214,...,50.194912000000002,False,False,"Вид услуги Сад, благоустройство Место оказания...",82b3686dd6ee49a3,"Копаем траншеи, сливные ямы, прокладка канализ...",114,0,0.0,0.0


In [16]:
import joblib

joblib.dump(vectorizer, "tfidf_train.pkl")

['tfidf_train.pkl']

In [17]:
# Финальный набор признаков: текст (TF-IDF-косинусы) и (рейтинг, цена, флаги).
# Геопризнаки (location_id, distance) и микрокатегория исключены — давали утечку. Во время обучения на первых
# шагах обучения давали метрику почти 1 - модель не училась, а заучивала ответы по расстоянию и метке локации.

feature_cols = [
    "item_rating",
    "item_rating_reviews_count",
    "item_price",
    "tfidf_cosine_title",
    "tfidf_cosine_desc",
    "search_is_delivery_search",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
]

# Целевая переменная (1/0)
target = "label"

#  приводим флаги к int: True/False - 1/0, чтобы XGBoost работал с числами.
for col in ["item_is_phone_hidden", "item_is_message_forbidden", "search_is_delivery_search"]:
    TrainData[col] = TrainData[col].astype(int)

# query_id: уникальный идентификатор запроса (текст, локация, фильтры).
# Нужен для группового разбиения — чтобы один запрос не попал и в train, и в val.
TrainData["query_id"] = TrainData.groupby(
    ["search_query", "search_location_id", "search_infm_params_text"]
).ngroup()


# GroupShuffleSplit разбивает по группам (query_id), а не по строкам.
# Все строки одного запроса попадают целиком в train или в val, что предотвращает утечки данных.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(TrainData, groups=TrainData["query_id"]))

train_part = TrainData.iloc[train_idx].reset_index(drop=True)
val_part = TrainData.iloc[val_idx].reset_index(drop=True)

X_train = train_part[feature_cols]
Y_train = train_part["label"]
X_test = val_part[feature_cols]
Y_test = val_part["label"]

# item_price может быть object (строки вместо чисел) — приводим к float.
# errors="coerce" преобразовывает нечисловые значения в NaN.
X_train["item_price"] = pd.to_numeric(X_train["item_price"], errors="coerce")
X_test["item_price"] = pd.to_numeric(X_test["item_price"], errors="coerce")

In [24]:
from xgboost import XGBClassifier

# XGBoost — градиентный бустинг над решающими деревьями.
# Я решил выбрать эту модель, поскольку она хорошо работает на табличных данных, устойчива к пропускам (NaN), быстра на CPU.

model = XGBClassifier(
    n_estimators= 500,
    learning_rate=0.05,
    max_depth=6,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    random_state=42,
    early_stopping_rounds=50,
    verbosity=0,
)

# Обучаем с валидацией: смотрим AUC на отложенной выборке.
model.fit(
    X_train, Y_train,
    eval_set=[(X_test, Y_test)],
    verbose=50,
)

[0]	validation_0-auc:0.93690
[50]	validation_0-auc:0.94046
[100]	validation_0-auc:0.94200
[150]	validation_0-auc:0.94303
[200]	validation_0-auc:0.94371
[250]	validation_0-auc:0.94418
[300]	validation_0-auc:0.94453
[350]	validation_0-auc:0.94482
[400]	validation_0-auc:0.94503
[450]	validation_0-auc:0.94523
[499]	validation_0-auc:0.94546


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [25]:
# Анализ результатов
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

preds = model.predict_proba(X_test)[:, 1]
pred_labels = (preds >= 0.5).astype(int)

print("AUC:", roc_auc_score(Y_test, preds))
print("Accuracy:", accuracy_score(Y_test, pred_labels))
print("Precision:", precision_score(Y_test, pred_labels))
print("Recall:", recall_score(Y_test, pred_labels))
print("F1:", f1_score(Y_test, pred_labels))

pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

AUC: 0.9454613266139592
Accuracy: 0.9346529329726279
Precision: 0.9461476492044666
Recall: 0.8524895909993648
F1: 0.8968801540036168


,feature,importance
4,tfidf_cosine_desc,0.646493
3,tfidf_cosine_title,0.330241
6,item_is_phone_hidden,0.007180
7,item_is_message_forbidden,0.004485
2,item_price,0.004115
1,item_rating_reviews_count,0.004089
0,item_rating,0.003398
5,search_is_delivery_search,0.000000


In [26]:
# Сохраняем модель и словарь на диск для повторного использования
joblib.dump(model, "xgb_model.pkl")
joblib.dump(vectorizer, "tfidf_train.pkl")

['tfidf_train.pkl']

In [27]:
# Выгружаем запросы без разметки
Queries = pd.read_parquet("benchmark_queries.parquet")

In [28]:
Location = DataItems.groupby("item_location_id")[
    ["item_longitude", "item_latitude"]
].median().reset_index()

In [29]:
import pandas as pd
import numpy as np

# Приводим координаты к float
DataItems["item_latitude"] = pd.to_numeric(DataItems["item_latitude"], errors="coerce").astype(float)
DataItems["item_longitude"] = pd.to_numeric(DataItems["item_longitude"], errors="coerce").astype(float)

Location = DataItems.groupby("item_location_id")[
    ["item_longitude", "item_latitude"]
].median().reset_index()

Location["item_latitude"] = pd.to_numeric(Location["item_latitude"], errors="coerce").astype(float)
Location["item_longitude"] = pd.to_numeric(Location["item_longitude"], errors="coerce").astype(float)

# Добавляем координаты запроса к Queries
query_coords = Location.rename(columns={
    "item_location_id": "search_location_id",
    "item_longitude": "search_longitude",
    "item_latitude": "search_latitude",
})

Queries = Queries.merge(query_coords, on="search_location_id", how="left")

Queries["search_latitude"] = pd.to_numeric(Queries["search_latitude"], errors="coerce").astype(float)
Queries["search_longitude"] = pd.to_numeric(Queries["search_longitude"], errors="coerce").astype(float)

# Формула гаверсинуса — расстояние между двумя точками на сфере по их широте и долготе.
# R = 6371 км — средний радиус Земли.
# Возвращает расстояние в километрах.
# Используется для оценки близости исполнителя к локации запроса.
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
from sklearn.preprocessing import normalize

# Приводим типы в DataItems
DataItems["item_price"] = pd.to_numeric(DataItems["item_price"], errors="coerce")
DataItems["item_is_phone_hidden"] = DataItems["item_is_phone_hidden"].astype(int)
DataItems["item_is_message_forbidden"] = DataItems["item_is_message_forbidden"].astype(int)
DataItems["item_title_raw"] = DataItems["item_title_raw"].fillna("").astype(str)
DataItems["item_description_raw"] = DataItems["item_description_raw"].fillna("").astype(str)

# Векторизуем все заголовки и описания
T_all = normalize(vectorizer.transform(DataItems["item_title_raw"]))
D_all = normalize(vectorizer.transform(DataItems["item_description_raw"]))

answers = []

for i, query in Queries.iterrows():
    # Фильтр по главной категории
    mask_cat = (DataItems["item_category_id"] == query["search_category"]).values
    idx_cat = np.where(mask_cat)[0]

    # Если в корпусе нет объявлений категории запроса - берём весь корпус
    # сортируем только по тексту (расстояние не используем).
    fallback = False
    if len(idx_cat) == 0:
        idx_cat = np.arange(len(DataItems))
        fallback = True

    # 2. Векторизуем запрос
    q_vec = normalize(vectorizer.transform([str(query["search_query"])]))

    # 3. TF-IDF-косинусы
    q_title = np.asarray(q_vec.multiply(T_all[idx_cat]).sum(axis=1)).ravel()
    q_desc = np.asarray(q_vec.multiply(D_all[idx_cat]).sum(axis=1)).ravel()

    # Отбор кандидатов
    if fallback:
        # Категории нет - только текст
        order = np.argsort(-q_title)
    elif not pd.isna(query["search_latitude"]):
        # Категория есть + координаты - по расстоянию
        dist = haversine(
            float(query["search_latitude"]), float(query["search_longitude"]),
            DataItems["item_latitude"].values[idx_cat],
            DataItems["item_longitude"].values[idx_cat],
        )
        order = np.argsort(dist)
    else:
        # Категория есть, координат нет — по тексту
        order = np.argsort(-q_title)

    # Топ-5000
    n_take = min(5000, len(order))
    top_local = order[:n_take]
    top_global = idx_cat[top_local]

    # Признаки
    cand = DataItems.iloc[top_global].copy()
    cand["tfidf_cosine_title"] = q_title[top_local]
    cand["tfidf_cosine_desc"] = q_desc[top_local]
    cand["search_is_delivery_search"] = int(query["search_is_delivery_search"])

    # Вероятности
    X = cand[feature_cols]
    preds = model.predict_proba(X)[:, 1]
    cand["score"] = preds

    # 8. Топ-50
    top = cand.nlargest(50, "score")["item_id"].tolist()

    answers.append({
        "query_id": query["query_id"],
        "answer": " ".join(top),
    })

# Сохраняем answer.csv
answer_df = pd.DataFrame(answers)
answer_df.to_csv("answer.csv", index=False)

print("Строк:", len(answer_df))
print("Пустых ответов:", (answer_df["answer"] == "").sum())
print(answer_df.head())

## Описание решения

### 1. Данные и признаки

**Данные:**
- `train.parquet` (497 673 строки) - обучающая выборка, пары «запрос - выбранное объявление».
- `benchmark_queries.parquet` (2 452 запроса) - тестовые запросы без разметки.
- `benchmark_items.parquet` (189 212 объявлений) - корпус для поиска кандидатов.

**Признаки:**

*Текстовые (TF-IDF-косинус):*
- `tfidf_cosine_title` - косинус между запросом и заголовком объявления.
- `tfidf_cosine_desc` - косинус между запросом и описанием.

*Обоснование:* текст - основной сигнал релевантности. Пользователь ищет «автоподбор» — выбирает объявление, где это слово есть в заголовке. TF-IDF учитывает частоту и редкость слов, косинус даёт интерпретируемую метрику от 0 до 1. По важности признаков текст дал 98% вклада.

*Числовые данные объявления:*
- `item_rating`, `item_rating_reviews_count` - рейтинг и число отзывов.
- `item_price` - цена.

*Обоснование:* качество исполнителя и бюджет важны для выбора.

*Бинарные (флаги):*
- `search_is_delivery_search` - флаг поиска с доставкой.
- `item_is_phone_hidden` - телефон скрыт.
- `item_is_message_forbidden` - сообщения запрещены.

*Почему:* удобство связи влияет на выбор.

**Признаки, которые исключил:**
- `search_location_id`, `item_location_id` - утечка: модель запоминала ID локаций, а не училась различать релевантность.
- `distance` - та же утечка: позитивы в train имели медиану 2.9 км, негативы - 1213 км. Модель выучила порог, а не смысл.
- `item_microcat_id` - утечка: негативы генерировались из той же микрокатегории, модель запоминала ID.
- `search_category`, `item_category_id` - константа 114 у всех строк, бесполезны как признак.

После удаления геопризнаков AUC упал с 0.999 до 0.9445.

### 2. Модели и алгоритмы

**Модель:** `XGBoostClassifier` (градиентный бустинг над решающими деревьями).

**Параметры:**
- `n_estimators=500`, `learning_rate=0.05`, `max_depth=6`.
- `tree_method="hist"` - ускорение на больших данных.
- `early_stopping_rounds=50` - остановка при отсутствии улучшений.
- `verbosity=0` - отключение логов.

**Почему XGBoost:**
- Хорошо работает на табличных данных.
- Устойчив к пропускам (`NaN` в `item_price`, `item_rating`).
- Быстрый на CPU, не требует GPU.
- Встроенная регуляризация, устойчив к переобучению.

**Альтернативы:**
- `CatBoost` - лучше с категориальными, но их не осталось после удаления ID.
- `LightGBM` - быстрее, но менее стабилен на маленьких данных.
- `RandomForest` - уступает бустингу на табличных данных.
- Логистическая регрессия - использовал бы как baseline, но XGBoost дал лучший AUC.

**Генерация негативов:**
В `train.parquet` только позитивы. Для обучения классификатора добавил негативы:
- Для каждого запроса - 2 случайных объявления из той же главной категории (114), но не выбранные пользователем.
- `label = 1` для позитивов, `label = 0` для негативов.
- Соотношение 1:2 (497к позитивов, 995к негативов).

**Разбиение train/val:**
`GroupShuffleSplit` по `query_id` (запрос + локация + фильтры) — 80/20. Все строки одного запроса попадают целиком в train или в val, без утечки.

**Двухэтапная архитектура:**
1. Фильтр по главной категории.
2. Предварительный отбор кандидатов:
   - если есть координаты запроса - топ-2000 по расстоянию (гаверсинус);
   - если нет - топ-2000 по `tfidf_cosine_title`.
3. ML-скоринг по отобранным кандидатам.
4. Топ-50 по вероятности.

**Обоснование:** перебирать 2 452 на 189 212 - около 464 млн пар напрямую дорого. Предварительный отбор сокращает до 2 452 * 2000 около 5 млн пар - быстро и практично.

### 3. Проверка качества

**Метрики на валидации:**
- AUC - 0.9445.
- Accuracy - 0.9345.
- Precision - 0.9457.
- Recall - 0.8526.
- F1 - 0.8967.

**Как считал:**
Разбил `TrainData` через `GroupShuffleSplit` по `query_id` - 80% train, 20% val. Метрики считал на val-части через `sklearn.metrics`.

**Что проверял:**
1. Утечка. Проверил пересечение `query_id` между train и val — пусто. Проверил важность признаков — геопризнаки давали 67% важности (утечка), убрал.
2. Дисбаланс классов. Соотношение 1:2 - не критично, но проверил.
3. Пропуски. `item_price`, `item_rating` содержат `NaN`. XGBoost обрабатывает их нативно.
4. Типы данных. Привёл флаги к `int`, `item_price` к `float`.
5. Воспроизводимость. Все `random_state=42`, `random.seed(42)`. Модель сохраняется в `.pkl`.
6. Формат `answer.csv`. Проверил: 2452 строки, нет дублей `query_id`, до 50 `item_id` в строке, все ID по 16 символов, все существуют в корпусе.


### 4. Ошибки и решения

**Ошибка 1. Утечка через геопризнаки.**

AUC = 0.999 на валидации, на первой итерации - 0.99.
*Анализ:* важность признаков показала, что `search_location_id` (27%) и `item_location_id` (25%) доминируют. Модель запоминала ID локаций. Распределение `distance`: позитивы — медиана 2.9 км, негативы — 1213 км. Модель выучила порог «близко/далеко».
*Решение:* убрал `location_id`, `distance`, координаты. AUC упал до 0.9445 — честный сигнал.

**Ошибка 2. Утечка через микрокатегорию.**

AUC = 0.998 без геопризнаков.
*Анализ:* `item_microcat_id` давал 15% важности. Негативы генерировались из той же микрокатегории, что и позитив. Модель запоминала ID микрокатегории.
*Решение:* убрал `item_microcat_id` из признаков.

**Ошибка 3. Дисбаланс негативов по локации.**

модель легко разделяла позитивы и негативы по расстоянию.
*Анализ:* негативы генерировались случайно из микрокатегории, без учёта локации. Позитивы близко, негативы далеко.
*Решение:* убрал `distance` из признаков.

**Ошибка 4. `ArrowMemoryError` при split.**

падение при `TrainData.iloc[train_idx]` на 1.5 млн строк.
*Анализ:* длинные текстовые колонки (`item_description_raw`) занимали гигабайты.
*Решение:* удалил текстовые колонки после расчёта TF-IDF. Загружал только нужные колонки из parquet.

**Ошибка 5. `XGBoostError: verbosity exceed bound`.**

падение при `model.fit` и `model.predict_proba` с `verbosity=100`.
*Анализ:* баг XGBoost — `verbose=100` в `fit` записывался в глобальный конфиг, при восстановлении падал на проверке `[0,3]`.
*Решение:* использовал `verbose=50`, `verbosity=0`. Перезапустил ядро для сброса конфига.

**Ошибка 6. `AttributeError: 'decimal.Decimal' object has no attribute 'radians'`.**

падение при расчёте расстояния.
*Анализ:* parquet возвращал координаты типа `Decimal`, а `np.radians` работает только с числами.
*Решение:* привёл координаты к `float` через `pd.to_numeric(...).astype(float)`.

**Ошибка 7. Дисбаланс негативов в маленьких микрокатегориях.**

в некоторых микрокатегориях 1–2 объявления, негативов не хватает.
*Анализ:* медиана размера микрокатегории - 2, минимум - 1.
*Решение:* перешёл на негативы из главной категории (114), а не из микрокатегории. 187к кандидатов — хватает на всех.

**Ошибка 8. 
222 запроса категории 0 без кандидатов.**

пустые ответы для 222 запросов.
*Анализ:* в `DataItems` нет объявлений категории 0. Фильтр по категории отсекал всё.
*Решение:* fallback - если для категории запроса нет объявлений, беру весь корпус и сортирую по тексту. Пустых ответов стало 0.


#  Использованные библиотеки 
pandas, numpy - работа с данными

scikit-learn - TF-IDF, метрики, разбиение

xgboost - градиентный бустинг

joblib - сохранение модели

json - сохранение конфига



#  Установка зависимостей (если не установлены):

!pip install pandas numpy scikit-learn xgboost pyarrow joblib